# Grafos y modelos generativos

Redes de grafos (GCN), fusión condicional y generación multimodal (HCMG), inversión de gradiente.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

## Normalización adaptativa y fusión condicional (HCMG)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdaptiveNormalization(nn.Module):
    """
    Normalización Adaptativa basada en un vector de condición.
    Similar a SPADE o AdaIN, pero simplificado.
    """
    def __init__(self, num_features, cond_dim):
        super().__init__()
        self.norm = nn.InstanceNorm2d(num_features, affine=False) # Normalización sin parámetros aprendibles
        # Capas lineales para predecir los parámetros de escala (gamma) y bias (beta)
        self.gamma_proj = nn.Linear(cond_dim, num_features)
        self.beta_proj = nn.Linear(cond_dim, num_features)

    def forward(self, x, cond_vector):
        # x: activations [batch, channels, H, W]
        # cond_vector: conditioning vector [batch, cond_dim]

        out = self.norm(x) # Normalizar sin affine

        # Predecir gamma y beta a partir del vector de condición
        gamma = self.gamma_proj(cond_vector).unsqueeze(-1).unsqueeze(-1) # [batch, channels, 1, 1]
        beta = self.beta_proj(cond_vector).unsqueeze(-1).unsqueeze(-1)   # [batch, channels, 1, 1]

        # Aplicar escala y bias adaptativos
        out = out * (1 + gamma) + beta
        return out

class ConditionalFusionBlock(nn.Module):
    """
    Bloque de Fusión Condicional (BFC).
    Combina activaciones con vectores condicionales (imagen + texto).
    """
    def __init__(self, in_channels, out_channels, img_id_cond_dim, text_cond_dim):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # Normalización adaptativa para la condición de imagen/ID
        self.norm_img_id = AdaptiveNormalization(out_channels, img_id_cond_dim)
        # Normalización adaptativa para la condición de texto
        self.norm_text = AdaptiveNormalization(out_channels, text_cond_dim)

    def forward(self, x, img_id_vector, text_cond_vector):
        # x: mapa de características de la capa anterior
        # img_id_vector: vector de características de imagen + ID
        # text_cond_vector: vector de características de texto

        out = self.conv(x)

        # Aplicar normalización adaptativa secuencialmente
        # Primero la condición de imagen/ID, luego la de texto
        out = self.norm_img_id(out, img_id_vector)
        out = self.norm_text(out, text_cond_vector)

        out = self.relu(out)
        return out

class HCMG(nn.Module):
    """
    Generador Condicionado Multimodal y Jerárquico (HCMG).
    """
    def __init__(self, input_image_channels, output_image_channels,
                 img_id_cond_dim, text_cond_dim, base_channels=64):
        super().__init__()

        # Capa de entrada de píxeles
        self.initial_conv = nn.Conv2d(input_image_channels, base_channels, kernel_size=3, padding=1)

        # Bloques de Fusión Condicional
        # Podríamos tener BFCs en diferentes resoluciones o profundidades
        self.bfc1 = ConditionalFusionBlock(base_channels, base_channels * 2, img_id_cond_dim, text_cond_dim)
        self.downsample1 = nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1) # Downsample para aumentar features

        self.bfc2 = ConditionalFusionBlock(base_channels * 2, base_channels * 4, img_id_cond_dim, text_cond_dim)
        self.downsample2 = nn.Conv2d(base_channels * 4, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bfc3 = ConditionalFusionBlock(base_channels * 4, base_channels * 8, img_id_cond_dim, text_cond_dim)

        # Decodificador (para generar la imagen)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.gen_conv1 = ConditionalFusionBlock(base_channels * 4, base_channels * 2, img_id_cond_dim, text_cond_dim) # Re-inyectar condiciones aquí

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.gen_conv2 = ConditionalFusionBlock(base_channels, base_channels, img_id_cond_dim, text_cond_dim) # Re-inyectar condiciones aquí

        self.final_conv = nn.Conv2d(base_channels, output_image_channels, kernel_size=3, padding=1)
        self.tanh = nn.Tanh() # Para salida de imagen (-1 a 1)

    def forward(self, input_pixels, img_id_vector, text_cond_vector):
        # input_pixels: [batch, input_image_channels, H, W]
        # img_id_vector: [batch, img_id_cond_dim]
        # text_cond_vector: [batch, text_cond_dim]

        # Codificación inicial de píxeles
        x = self.initial_conv(input_pixels) # [batch, base_channels, H, W]

        # Paso por BFCs y downsampling
        x = self.bfc1(x, img_id_vector, text_cond_vector) # [batch, base_channels*2, H, W]
        x = self.downsample1(x) # [batch, base_channels*2, H/2, W/2]

        x = self.bfc2(x, img_id_vector, text_cond_vector) # [batch, base_channels*4, H/2, W/2]
        x = self.downsample2(x) # [batch, base_channels*4, H/4, W/4]

        x = self.bfc3(x, img_id_vector, text_cond_vector) # [batch, base_channels*8, H/4, W/4]

        # Decodificación y upsampling con re-inyección condicional
        x = self.upsample1(x)
        x = self.gen_conv1(x, img_id_vector, text_cond_vector) # [batch, base_channels*2, H/2, W/2]

        x = self.upsample2(x)
        x = self.gen_conv2(x, img_id_vector, text_cond_vector) # [batch, base_channels, H, W]

        # Capa de salida
        output_image = self.tanh(self.final_conv(x))
        return output_image

# --- Ejemplo de Uso ---
input_channels = 3 # Por ejemplo, una imagen de contorno o una imagen muy borrosa
output_channels = 3 # Salida de imagen RGB
img_id_cond_dim = 128 # Dimensión del vector de características de imagen + ID (ej. ResNet features + one-hot ID)
text_cond_dim = 768 # Dimensión de embedding de texto (ej. de un Transformer de texto como BERT o CLIP)

model = HCMG(input_channels, output_channels, img_id_cond_dim, text_cond_dim)
print(model)

# Crear inputs dummy
dummy_input_pixels = torch.randn(4, 3, 64, 64) # Batch de 4 imágenes 64x64
dummy_img_id_vector = torch.randn(4, img_id_cond_dim) # 4 vectores de condición de imagen
dummy_text_cond_vector = torch.randn(4, text_cond_dim) # 4 vectores de condición de texto

# Pasar por el modelo
output_image = model(dummy_input_pixels, dummy_img_id_vector, dummy_text_cond_vector)
print(f"\nForma de la imagen de salida: {output_image.shape}")

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Número total de parámetros: {total_params}")

## Red de grafos (GCN)

In [ ]:
# --- Configuración del entorno y PyTorch Geometric ---
# Descomentar y ejecutar en Colab si no tienes las librerías
import os
!pip install -q torch
os.environ['TORCH'] = torch.__version__
os.environ['PYTHONWARNINGS'] = "ignore"
!pip install torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt
import torch
import os
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv, GATConv
import numpy as np


# Código para el grafo de NetworkX (Introducción)
H = nx.DiGraph()

# Adding nodes
H.add_nodes_from([
  (0, {"color": "blue", "size": 250}),
  (1, {"color": "yellow", "size": 400}),
  (2, {"color": "orange", "size": 150}),
  (3, {"color": "red", "size": 600})
])

# Adding edges
H.add_edges_from([
  (0, 1),
  (1, 2),
  (1, 0),
  (1, 3),
  (2, 3),
  (3,0)
])

node_colors = nx.get_node_attributes(H, "color").values()
colors = list(node_colors)
node_sizes = nx.get_node_attributes(H, "size").values()
sizes = list(node_sizes)

# Plotting Graph (Directed)
nx.draw(H, with_labels=True, node_color=colors, node_size=sizes)
plt.title("Grafo Dirigido (H)")
plt.show() # Asegúrate de mostrar el primer grafo

G = H.to_undirected()
# Plotting Graph (Undirected)
nx.draw(G, with_labels=True, node_color=colors, node_size=sizes)
plt.title("Grafo No Dirigido (G)")
plt.show() # Asegúrate de mostrar el segundo grafo

# --- Carga del Dataset Planetoid Cora ---
dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())

print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.
print(data)


# --- Implementación del Modelo GCN (Graph Convolutional Network) ---
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GCNConv(dataset.num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model_gcn = GCN(hidden_channels=16)
print("Modelo GCN:")
print(model_gcn)


# --- Función para Visualización con t-SNE ---
# %matplotlib inline
def visualize(h, color, title="Visualización de Nodos"):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])
    plt.title(title)
    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

# --- Entrenamiento del Modelo GCN ---
optimizer_gcn = torch.optim.Adam(model_gcn.parameters(), lr=0.01, weight_decay=5e-4)
criterion_gcn = torch.nn.CrossEntropyLoss()

def train_gcn():
      model_gcn.train()
      optimizer_gcn.zero_grad()
      out = model_gcn(data.x, data.edge_index)
      loss = criterion_gcn(out[data.train_mask], data.y[data.train_mask])
      loss.backward()
      optimizer_gcn.step()
      return loss

def test_gcn():
      model_gcn.eval()
      out = model_gcn(data.x, data.edge_index)
      pred = out.argmax(dim=1)
      test_correct = pred[data.test_mask] == data.y[data.test_mask]
      test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
      return test_acc

print("\n--- Entrenamiento del Modelo GCN ---")
for epoch in range(1, 101):
    loss = train_gcn()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

test_acc_gcn = test_gcn()
print(f'Test Accuracy (GCN): {test_acc_gcn:.4f}')

model_gcn.eval()
out_gcn = model_gcn(data.x, data.edge_index)
visualize(out_gcn, color=data.y, title="Visualización de Nodos después de GCN")


# --- Implementación del Modelo GAT (Graph Attention Network) ---
class GAT(torch.nn.Module):
    def __init__(self, hidden_channels, heads):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GATConv(dataset.num_features, hidden_channels, heads)
        # La segunda capa toma 'heads * hidden_channels' como entrada
        self.conv2 = GATConv(heads * hidden_channels, dataset.num_classes, heads=1) # Generalmente la última capa de GAT usa heads=1 para la clasificación

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x) # Función de activación ELU es común en GAT
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# Nota: El código original tenía un print de GAT() antes de la definición completa, lo he movido.
model_gat = GAT(hidden_channels=8, heads=8)
print("\nModelo GAT:")
print(model_gat)

# --- Entrenamiento del Modelo GAT ---
optimizer_gat = torch.optim.Adam(model_gat.parameters(), lr=0.005, weight_decay=5e-4)
criterion_gat = torch.nn.CrossEntropyLoss()

def train_gat():
      model_gat.train()
      optimizer_gat.zero_grad()
      out = model_gat(data.x, data.edge_index)
      loss = criterion_gat(out[data.train_mask], data.y[data.train_mask])
      loss.backward()
      optimizer_gat.step()
      return loss

def test_gat(mask):
      model_gat.eval()
      out = model_gat(data.x, data.edge_index)
      pred = out.argmax(dim=1)
      correct = pred[mask] == data.y[mask]
      acc = int(correct.sum()) / int(mask.sum())
      return acc

val_acc_all_gat = []
test_acc_all_gat = []

print("\n--- Entrenamiento del Modelo GAT ---")
for epoch in range(1, 101):
    loss = train_gat()
    val_acc = test_gat(data.val_mask)
    test_acc = test_gat(data.test_mask)
    val_acc_all_gat.append(val_acc)
    test_acc_all_gat.append(test_acc)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}')


# --- Visualización de resultados de GAT ---
plt.figure(figsize=(12,8))
plt.plot(np.arange(1, len(val_acc_all_gat) + 1), val_acc_all_gat, label='Validation accuracy', c='blue')
plt.plot(np.arange(1, len(test_acc_all_gat) + 1), test_acc_all_gat, label='Testing accuracy', c='red')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('GATConv Accuracy over Epochs')
plt.legend(loc='lower right', fontsize='x-large')
# plt.savefig('gat_loss.png') # Descomentar para guardar la imagen
plt.show()

model_gat.eval()
out_gat = model_gat(data.x, data.edge_index)
visualize(out_gat, color=data.y, title="Visualización de Nodos después de GAT")

In [ ]:
import torch
from torch_geometric.data import Data

# --- Definir las características de tus "neuronas" ---

# Ejemplo:
# Neurona 0: Capa de Entrada, posición (0.0, 1.0)
# Neurona 1: Capa de Entrada, posición (0.0, 2.0)
# Neurona 2: Capa Oculta, posición (1.0, 1.5)
# Neurona 3: Capa de Salida, posición (2.0, 1.5)

# Características para data.x:
# Columna 0: Tipo de neurona (ej. 0=entrada, 1=oculta, 2=salida) - One-hot encoding podría ser mejor
# Columna 1: Posición X
# Columna 2: Posición Y
# Columna 3: Otra característica (ej. Valor de activación inicial si aplica, o un ID embebido)
# ... puedes agregar más características relevantes para tu problema

# Usaremos un encoding simple para el tipo (0, 1, 2). Para un uso real, one-hot encoding sería mejor.
# Por ejemplo, si el tipo de neurona es una característica categórica, puedes usar one-hot encoding:
# Tipo de neurona: [es_entrada, es_oculta, es_salida]
# 0: entrada -> [1, 0, 0]
# 1: oculta  -> [0, 1, 0]
# 2: salida  -> [0, 0, 1]

# Ejemplo con one-hot encoding para el tipo de neurona:
# Dimensión de características: 3 (para tipo) + 2 (para posición) + 1 (para otra_info) = 6
features_neuronas = torch.tensor([
    # Tipo       Pos X  Pos Y  Otra info
    [1.0, 0.0, 0.0, 0.0,  1.0,   0.5], # Neurona 0 (Entrada)
    [1.0, 0.0, 0.0, 0.0,  2.0,   0.8], # Neurona 1 (Entrada)
    [0.0, 1.0, 0.0, 1.0,  1.5,   0.2], # Neurona 2 (Oculta)
    [0.0, 0.0, 1.0, 2.0,  1.5,   0.1]  # Neurona 3 (Salida)
], dtype=torch.float)

# --- Definir las conexiones (aristas) entre tus "neuronas" ---
# Las conexiones aquí serían las conexiones de la red neuronal tradicional.
# Ejemplo:
# De Neurona 0 (entrada) a Neurona 2 (oculta)
# De Neurona 1 (entrada) a Neurona 2 (oculta)
# De Neurona 2 (oculta) a Neurona 3 (salida)

edge_index_neuronas = torch.tensor([
    [0, 1, 2], # Origen de la conexión (índices de neuronas)
    [2, 2, 3]  # Destino de la conexión (índices de neuronas)
], dtype=torch.long)

# --- Crear el objeto Data de PyTorch Geometric ---
# Si tu objetivo es clasificar estas "neuronas" (ej. en tipos), podrías añadir data.y
# Por ejemplo, si la tarea es clasificar la "etapa" de la neurona (entrada, oculta, salida)
labels_neuronas = torch.tensor([0, 0, 1, 2], dtype=torch.long) # 0=entrada, 1=oculta, 2=salida

# Crear el objeto Data
data_red_neuronal_como_grafo = Data(
    x=features_neuronas,
    edge_index=edge_index_neuronas,
    y=labels_neuronas # Opcional, si tienes etiquetas
)

print(data_red_neuronal_como_grafo)
print(f"Características de las neuronas (data.x):\n{data_red_neuronal_como_grafo.x}")
print(f"Conexiones de las neuronas (data.edge_index):\n{data_red_neuronal_como_grafo.edge_index}")
print(f"Etiquetas de las neuronas (data.y):\n{data_red_neuronal_como_grafo.y}")

In [ ]:
# Definición de la neurona espacial
class NeuronaEspacial:
    def __init__(self, id_neurona, dimensiones, posicion):
        self.id = id_neurona
        self.dimensiones = dimensiones  # Ejemplo: (alto, ancho, profundidad)
        self.posicion = posicion      # Ejemplo: (x, y, z)
        self.activacion = 0.0         # Nivel de activación de la neurona
        self.conexiones_entrantes = [] # Lista de (neurona_origen, peso)
        self.conexiones_salientes = [] # Lista de (neurona_destino, peso)

    # Método para recibir una señal y actualizar la activación
    def recibir_senal(self, senal_entrante):
        # En un modelo simple, la activación puede ser la suma ponderada de las señales
        self.activacion += senal_entrante

    # Método para enviar una señal a otras neuronas
    def enviar_senal(self):
        senales_salientes = {}
        for neurona_destino, peso in self.conexiones_salientes:
            # La señal enviada es la activación de esta neurona multiplicada por el peso
            senales_salientes[neurona_destino] = self.activacion * peso
        return senales_salientes

# --- Pseudocódigo de un experimento con neuronas espaciales ---

# 1. Inicialización del espacio y las neuronas
espacio = {} # Un diccionario para almacenar las neuronas, quizás indexadas por ID o posición
lista_neuronas = []

# Crear algunas neuronas con diferentes dimensiones y posiciones
# Supongamos 3 dimensiones (x, y, z)
neurona1 = NeuronaEspacial(1, (10, 10, 10), (0, 0, 0))
neurona2 = NeuronaEspacial(2, (5, 5, 5), (10, 5, 2))
neurona3 = NeuronaEspacial(3, (8, 8, 8), (2, 8, 12))

lista_neuronas.append(neurona1)
lista_neuronas.append(neurona2)
lista_neuronas.append(neurona3)

# 2. Establecer conexiones entre neuronas (ejemplo simple)
# Conectar neurona1 a neurona2 con un peso de 0.7
neurona1.conexiones_salientes.append((neurona2, 0.7))
neurona2.conexiones_entrantes.append((neurona1, 0.7))

# Conectar neurona2 a neurona3 con un peso de 0.5
neurona2.conexiones_salientes.append((neurona3, 0.5))
neurona3.conexiones_entrantes.append((neurona2, 0.5))

# 3. Proceso de activación (simulación)
print("--- Inicio de la simulación ---")

# Dar una señal inicial a una neurona
print(f"Activación inicial de Neurona 1: {neurona1.activacion}")
neurona1.recibir_senal(1.0) # Neurona 1 recibe una señal de 1.0
print(f"Activación de Neurona 1 después de recibir señal: {neurona1.activacion}\n")

# Neurona 1 envía señales
senales_de_neurona1 = neurona1.enviar_senal()
for destino, senal in senales_de_neurona1.items():
    print(f"Neurona 1 envía {senal:.2f} a Neurona {destino.id}")
    destino.recibir_senal(senal) # La neurona destino recibe la señal
print(f"Activación de Neurona 2 después de recibir señal: {neurona2.activacion}\n")

# Neurona 2 envía señales
senales_de_neurona2 = neurona2.enviar_senal()
for destino, senal in senales_de_neurona2.items():
    print(f"Neurona 2 envía {senal:.2f} a Neurona {destino.id}")
    destino.recibir_senal(senal)
print(f"Activación de Neurona 3 después de recibir señal: {neurona3.activacion}\n")

print("--- Fin de la simulación ---")

# 4. Observación de los estados finales
print("\nEstados finales de las neuronas:")
for neurona in lista_neuronas:
    print(f"Neurona {neurona.id}: Posición={neurona.posicion}, Dimensiones={neurona.dimensiones}, Activación={neurona.activacion:.2f}")

# Posibles extensiones para un experimento:
# - Implementar funciones de activación (ReLU, sigmoide, etc.)
# - Mecanismos de propagación de señales más complejos (propagación en el tiempo, decaimiento)
# - Reglas de aprendizaje que modifiquen los pesos de las conexiones
# - Visualización de las neuronas en su espacio tridimensional
# - Interacciones basadas en la proximidad espacial

In [ ]:
Un# prompt: ## Seudocódigo para Neuronas Espaciales (sin capas explícitas)
# # 1. Definición de la Neurona
# Clase Neurona:
#     Atributos:
#         id: Identificador único de la neurona
#         posicion: Una tupla o lista que representa las coordenadas (x, y, z, ...)
#         estado_activacion: Valor numérico (ej. 0 o 1, o un valor continuo)
#         conexiones_salientes: Lista de tuplas (neurona_destino, peso)
#         umbral_activacion: Valor que debe superar la suma ponderada para activarse
#     Métodos:
#         __init__(id, posicion, umbral):
#             Inicializar id, posicion, umbral_activacion
#             estado_activacion = 0
#             conexiones_salientes = []
#         añadir_conexion(neurona_destino, peso):
#             Añadir (neurona_destino, peso) a conexiones_salientes
#         calcular_activacion(entradas_recibidas):
#             # entradas_recibidas es un diccionario {neurona_origen_id: valor_señal}
#             suma_ponderada = 0
#             Para cada neurona_origen_id, valor_señal en entradas_recibidas:
#                 # Aquí se podría considerar la distancia espacial entre neuronas
#                 # para ajustar el peso o la influencia de la señal.
#                 # Por simplicidad, asumimos que entradas_recibidas ya maneja esto
#                 # o es una consideración a aplicar en la propagación.
#                 suma_ponderada = suma_ponderada + valor_señal
#             Si suma_ponderada > umbral_activacion:
#                 estado_activacion = funcion_activacion(suma_ponderada)
#             Sino:
#                 estado_activacion = 0 # O un valor de reposo
#             Retornar estado_activacion
#         propagar_señal():
#             # Distribuye el estado de activación a las neuronas conectadas
#             señales_a_enviar = []
#             Para cada conexion en conexiones_salientes:
#                 neurona_destino, peso = conexion
#                 señal = estado_activacion * peso
#                 Añadir (neurona_destino.id, señal)

# Implementación de la clase Neurona Espacial
class NeuronaEspacial:
    def __init__(self, id_neurona, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        self.id = id_neurona
        self.posicion = posicion  # Tupla o lista (ej. (x, y, z))
        self.dimensiones = dimensiones # Tupla o lista (ej. (alto, ancho, profundidad))
        self.activacion = 0.0  # Valor numérico del estado de activación
        self.umbral_activacion = umbral_activacion # Umbral para activarse
        self.tipo = tipo # "entrada", "oculta", "salida"

        self.conexiones_salientes = [] # Lista de tuplas (neurona_destino, peso)
        self.conexiones_entrantes = [] # Opcional: lista de (neurona_origen, peso)

    def añadir_conexion(self, neurona_destino, peso=1.0):
        """Añade una conexión saliente a otra neurona."""
        self.conexiones_salientes.append((neurona_destino, peso))

    def recibir_senal(self, valor_senal):
        """Recibe una señal entrante y la suma a la activación."""
        self.activacion += valor_senal

    def calcular_activacion(self, entradas_recibidas=None):
        """
        Calcula la activación basada en las señales recibidas.
        En un modelo simple, puede ser solo una función de activación.
        En modelos más complejos, puede involucrar umbrales o dinámicas.
        """
        # Para este pseudocódigo, asumimos que 'recibir_senal' ya sumó las entradas.
        # Si queremos un umbral, lo aplicamos aquí.
        if self.activacion > self.umbral_activacion:
             # Ejemplo simple de función de activación (relu)
             self.activacion = max(0, self.activacion)
             # O una sigmoide si quieres salida entre 0 y 1
             # self.activacion = 1 / (1 + math.exp(-self.activacion))
        else:
            self.activacion = 0.0 # Si no supera el umbral, se desactiva (o vuelve a un estado base)

        return self.activacion


    def propagar_señal(self):
        """
        Propaga la señal de activación a las neuronas conectadas.
        Retorna una lista de (neurona_destino, señal_enviada).
        """
        senales_a_enviar = []
        if self.activacion > 0: # Solo propaga si está activada
            for neurona_destino, peso in self.conexiones_salientes:
                señal = self.activacion * peso
                senales_a_enviar.append((neurona_destino, señal))
        return senales_a_enviar

# --- Implementación de un Espacio con Neuronas Espaciales ---
class EspacioNeuronal:
    def __init__(self):
        self.neuronas = {} # Diccionario {id_neurona: objeto_neurona}
        self.ultima_id = 0

    def añadir_neurona(self, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        """Añade una nueva neurona al espacio."""
        self.ultima_id += 1
        new_neurona = NeuronaEspacial(self.ultima_id, posicion, dimensiones, umbral_activacion, tipo)
        self.neuronas[self.ultima_id] = new_neurona
        return new_neurona

    def conectar_neuronas(self, id_origen, id_destino, peso=1.0):
        """Establece una conexión de origen a destino."""
        if id_origen in self.neuronas and id_destino in self.neuronas:
            neurona_origen = self.neuronas[id_origen]
            neurona_destino = self.neuronas[id_destino]
            neurona_origen.añadir_conexion(neurona_destino, peso)
            # Opcional: añadir conexión entrante si el modelo es bidireccional o lo necesita
            # neurona_destino.conexiones_entrantes.append((neurona_origen, peso))
        else:
            print(f"Error: Una o ambas neuronas ({id_origen}, {id_destino}) no existen.")

    def simular_paso_tiempo(self, senales_externas={}):
        """
        Simula un paso de tiempo en el espacio neuronal.
        Procesa señales externas, calcula activaciones y propaga señales.
        senales_externas: {id_neurona_destino: valor_senal}
        """
        # Paso 1: Aplicar señales externas y resetear activación (o decaerla)
        # Resetear la activación para un modelo no recurrente simple
        for neurona in self.neuronas.values():
             neurona.activacion = 0.0

        # Aplicar señales externas
        for id_destino, senal in senales_externas.items():
            if id_destino in self.neuronas:
                self.neuronas[id_destino].recibir_senal(senal)
            else:
                print(f"Advertencia: Señal externa para neurona {id_destino} que no existe.")


        # Paso 2: Recolectar todas las señales a propagar desde neuronas activas del paso anterior (o entradas directas)
        # En este modelo simple, las "señales recibidas" son directas del paso anterior o externas.
        # Las entradas a 'calcular_activacion' en el pseudocódigo original implicaban recolectar señales
        # EN este paso ANTES de recalcular la activación. Adaptamos el flujo:
        # Recolectamos señales de propagación *basadas en la activación ya actualizada por señales externas*
        # Si hubiera activación previa que decae, la consideraríamos aquí.

        senales_propagadas = []
        for neurona in self.neuronas.values():
            # Calcula la activación final después de recibir entradas (externas o de propagación previa)
            # En este simple ejemplo, la 'recibir_senal' ya sumó. Si tuvieras dinámica, la aplicarías aquí.
            # Por simplicidad, asumimos que la activación ya refleja las entradas sumadas.
            # Llama a calcular_activacion para aplicar umbral, etc.
            neurona.calcular_activacion() # Aplica umbral/func.activación a la activación actual

            # Ahora, si la neurona está activada, prepara sus señales de salida
            if neurona.activacion > 0:
                 senales_propagadas.extend(neurona.propagar_señal())


        # Paso 3: Distribuir las señales propagadas a las neuronas destino en el *próximo* paso de tiempo
        # Para simular en un solo paso, podemos aplicarlas inmediatamente, pero conceptualmente son "entradas para el próximo ciclo".
        # Como nuestro modelo simple reinicia activación cada paso, necesitamos acumular las entradas para la próxima ronda
        # o modificar la activación de destino DENTRO de este paso (si la propagación es instantánea)
        # Optamos por la propagación instantánea dentro del mismo paso para simplificar:

        # Reiniciar activaciones para el próximo ciclo antes de aplicar señales propagadas
        # (Este reset dependerá de la dinámica de tu modelo - decaimiento, etc.)
        # En este ejemplo, la propagación es instantánea dentro del mismo paso, así que no reseteamos AHORA.
        # Las señales propagadas de este paso se aplican *inmediatamente* a la activación de los destinos.

        # Ahora, distribuir las señales propagadas
        senales_recibidas_en_este_paso = {} # Acumula señales por destino
        for neurona_destino, senal in senales_propagadas:
             if neurona_destino.id not in senales_recibidas_en_este_paso:
                 senales_recibidas_en_este_paso[neurona_destino.id] = 0
             senales_recibidas_en_este_paso[neurona_destino.id] += senal

        # Aplicar las señales acumuladas a las neuronas destino
        for id_destino, total_senal_recibida in senales_recibidas_en_este_paso.items():
             self.neuronas[id_destino].recibir_senal(total_senal_recibida)


        # Después de aplicar TODAS las señales, las activaciones están actualizadas.
        # El estado de las neuronas al final de esta función es el resultado del paso de simulación.

    def get_estado(self):
        """Retorna un resumen del estado actual de las neuronas."""
        estado = {}
        for id_neurona, neurona in self.neuronas.items():
            estado[id_neurona] = {
                "posicion": neurona.posicion,
                "dimensiones": neurona.dimensiones,
                "activacion": neurona.activacion,
                "tipo": neurona.tipo
            }
        return estado

# --- Ejemplo de uso del EspacioNeuronal ---

# Inicializar un espacio
espacio = EspacioNeuronal()

# Añadir algunas neuronas espaciales
n1 = espacio.añadir_neurona(posicion=(0, 0, 0), dimensiones=(10, 10, 10), tipo="entrada", umbral_activacion=0.1)
n2 = espacio.añadir_neurona(posicion=(5, 5, 5), dimensiones=(8, 8, 8), tipo="oculta", umbral_activacion=0.3)
n3 = espacio.añadir_neurona(posicion=(10, 10, 10), dimensiones=(5, 5, 5), tipo="salida", umbral_activacion=0.5)
n4 = espacio.añadir_neurona(posicion=(5, 0, 10), dimensiones=(6, 6, 6), tipo="oculta", umbral_activacion=0.2)


# Conectar las neuronas
espacio.conectar_neuronas(n1.id, n2.id, peso=0.8)
espacio.conectar_neuronas(n1.id, n4.id, peso=0.6)
espacio.conectar_neuronas(n2.id, n3.id, peso=0.7)
espacio.conectar_neuronas(n4.id, n3.id, peso=0.9)


# Simular pasos de tiempo
print("--- Simulación del Espacio Neuronal ---")

# Paso 1: Aplicar señal externa a la neurona de entrada 1
print("\nPaso 1: Señal externa a Neurona 1")
espacio.simular_paso_tiempo(senales_externas={n1.id: 1.5})
estado_paso1 = espacio.get_estado()
for id, data in estado_paso1.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

# Paso 2: Simular otro paso (las activaciones se basan en el estado final del paso 1)
print("\nPaso 2: Simulación (propagación interna)")
# No hay señales externas en este paso, solo propagación interna
espacio.simular_paso_tiempo(senales_externas={})
estado_paso2 = espacio.get_estado()
for id, data in estado_paso2.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

# Paso 3: Simular otro paso (propagación)
print("\nPaso 3: Simulación (propagación)")
espacio.simular_paso_tiempo(senales_externas={})
estado_paso3 = espacio.get_estado()
for id, data in estado_paso3.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

print("\n--- Fin de la simulación ---")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
from sklearn.decomposition import PCA # Assuming PCA import is intended based on ipython-input-14

# Supongamos que tienes tus neuronas en una lista
# Se asume que la clase NeuronaEspacial está definida en celdas anteriores y es accesible.
# Si no, debes incluir su definición aquí.
# from . import NeuronaEspacial # Ejemplo si estuviera en otro archivo, no aplica en notebook

# Re-definiendo NeuronaEspacial just in case it's not available from previous cells
class NeuronaEspacial:
    def __init__(self, id_neurona, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        self.id = id_neurona
        self.posicion = posicion  # Tupla o lista (ej. (x, y, z))
        self.dimensiones = dimensiones # Tupla o lista (ej. (alto, ancho, profundidad))
        self.activacion = 0.0  # Valor numérico del estado de activación
        self.umbral_activacion = umbral_activacion # Umbral para activarse
        self.tipo = tipo # "entrada", "oculta", "salida"
        self.conexiones_salientes = []
        self.conexiones_entrantes = []


neuronas = [
    NeuronaEspacial(1, (0, 0, 0), (10, 10, 10), tipo="entrada"),
    NeuronaEspacial(2, (10, 5, 2), (5, 5, 5), tipo="oculta"),
    NeuronaEspacial(3, (2, 8, 12), (8, 8, 8), tipo="salida")
]

# Opcional: darles activaciones distintas para simular estados
neuronas[0].activacion = 1.0
neuronas[1].activacion = 0.7
neuronas[2].activacion = 0.35

# Codificar tipo como valor numérico
tipo_map = {"entrada": 0, "oculta": 1, "salida": 2}

# Vectorizar cada neurona
features = []
for n in neuronas:
    x, y, z = n.posicion
    alto, ancho, prof = n.dimensiones
    tipo_val = tipo_map.get(n.tipo, 1)
    features.append([x, y, z, alto, ancho, prof, n.activacion, tipo_val])

# Convertir a array para t-SNE
X = np.array(features)

# Aplicar t-SNE para reducir a 2D
# CORRECCIÓN: perplexity debe ser menor que el número de muestras (3).
# Cambiamos perplexity de 3 a 1 (o 2 también sería válido).
tsne = TSNE(n_components=2, perplexity=1, learning_rate=200, n_iter=1000, random_state=42)
X_embedded = tsne.fit_transform(X)

# Visualizar
plt.figure(figsize=(8, 6))
for i, coord in enumerate(X_embedded):
    tipo = neuronas[i].tipo
    plt.scatter(coord[0], coord[1], label=f"Neurona {neuronas[i].id} ({tipo})")
    plt.text(coord[0]+0.5, coord[1], f"ID {neuronas[i].id}", fontsize=9)

plt.title("Visualización 2D de Neuronas Espaciales con t-SNE")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.grid(True)
plt.show()

# PCA visualization (kept from original cell)
# Ensure X is defined from the features list above
pca = PCA(n_components=2)
X_pca_embedded = pca.fit_transform(X) # Using a different variable name for PCA results

plt.figure(figsize=(8, 6))
for i, coord in enumerate(X_pca_embedded):
    tipo = neuronas[i].tipo
    plt.scatter(coord[0], coord[1], label=f"Neurona {neuronas[i].id} ({tipo})")
    plt.text(coord[0]+0.5, coord[1], f"ID {neuronas[i].id}", fontsize=9)

plt.title("Visualización 2D de Neuronas Espaciales con PCA")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Ensure the built-in id function is not shadowed
# Check if 'id' is currently an integer in the global scope and remove it if it is.
# This is a protective measure against lingering variable assignments.
if 'id' in globals() and isinstance(globals()['id'], int):
    print("Warning: Found shadowed built-in 'id'. Removing the variable.")
    del globals()['id']

# Re-defining necessary imports and classes for this specific visualization part
# (This is for self-containment; in a real notebook, you might only need the last part)
try:
    from sklearn.datasets import make_circles
    from sklearn.preprocessing import StandardScaler
except ImportError:
    print("Please install scikit-learn: pip install scikit-learn")
    # In a notebook, you could try: !pip install scikit-learn
    # Or raise the error if installation is required.
    # For this fix, we assume it's installed based on the context.
    pass # Or raise ImportError("scikit-learn is required.")


# Assuming NeuronaResonante class exists from previous cell output

def sigmoide(x):
    """Función de activación Sigmoide."""
    # Use math.exp for scalar inputs which is generally faster for single values
    return 1 / (1 + math.exp(-x))

class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0, bias_inicial=None):
        self.entradas = []
        self.pesos = []
        self.bias = bias_inicial if bias_inicial is not None else np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0
        # Ensure we are calling the built-in id function by explicitly using __builtins__.id
        # This prevents shadowing issues.
        self.id = __builtins__.id(self) # Use built-in id() for unique object identifier


    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            # Use np.random.uniform which was likely intended based on previous code
            peso = np.random.uniform(-1, 1)
        self.pesos.append(peso)


    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            # Apply sigmoid only if activated
            self.salida = sigmoide(total)
        else:
            self.salida = 0.0 # Output is zero or base state if not activated above threshold


def crear_red(num_neuronas_entrada, num_neuronas_ocultas,
              fuerza_ocultas, amplificacion_ocultas,
              fuerza_salida, amplificacion_salida):

    # Ensure np.random.randn is used for bias initialization as in the original code
    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(num_neuronas_entrada)]
    ocultas = [NeuronaResonante(fuerza=fuerza_ocultas, amplificacion=amplificacion_ocultas)
               for _ in range(num_neuronas_ocultas)]
    salida = NeuronaResonante(fuerza=fuerza_salida, amplificacion=amplificacion_salida)

    for h in ocultas:
        for e in entrada:
            h.conectar(e)

    for h in ocultas:
        salida.conectar(h)

    return entrada, ocultas, salida


def evaluar_red(entrada, ocultas, salida, X_data, y_data):
    errores = []
    for xi, yi in zip(X_data, y_data):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        # Use binary cross-entropy like error if output is sigmoid [0,1]
        # For simplicity, keeping squared error from original cell
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)


def copiar_estado_red(entrada_src, ocultas_src, salida_src):
    # Pass existing bias to the new NeuronaResonante instance
    entrada_dst = [NeuronaResonante(n.fuerza, n.amplificacion, bias_inicial=n.bias) for n in entrada_src]
    ocultas_dst = [NeuronaResonante(n.fuerza, n.amplificacion, bias_inicial=n.bias) for n in ocultas_src]
    salida_dst = NeuronaResonante(salida_src.fuerza, salida_src.amplificacion, bias_inicial=salida_src.bias)

    for i, h_dst in enumerate(ocultas_dst):
        for j, e_dst in enumerate(entrada_dst):
            # Ensure weights are copied correctly
            h_dst.conectar(e_dst, ocultas_src[i].pesos[j])

    for i, h_dst in enumerate(ocultas_dst):
        # Ensure weights are copied correctly
        salida_dst.conectar(h_dst, salida_src.pesos[i])

    return entrada_dst, ocultas_dst, salida_dst


def entrenar_con_hiperparametros(X, y, config):
    best_loss = float('inf')
    best_red_state = None

    entrada, ocultas, salida = crear_red(
        config['num_neuronas_entrada'],
        config['num_neuronas_ocultas'],
        config['fuerza_ocultas'],
        config['amplificacion_ocultas'],
        config['fuerza_salida'],
        config['amplificacion_salida']
    )

    print(f"Starting training with {config['n_generaciones']} generations...")

    for gen in range(config['n_generaciones']):
        current_loss = evaluar_red(entrada, ocultas, salida, X, y)

        if current_loss < best_loss:
            best_loss = current_loss
            # Copy the current state, not the best found so far, if the goal is to continue mutating the improving network
            # If the goal is a simple evolutionary algorithm saving the best, then the original logic is fine.
            # Assuming the original logic of saving the best and then mutating the current is intended:
            best_red_state = copiar_estado_red(entrada, ocultas, salida) # This line seems correct based on original intent


        # Mutate the current network state (entrada, ocultas, salida)
        for neurona_list in [ocultas, [salida]]: # Exclude input neurons from mutation
            for neurona in neurona_list:
                # Mutate bias
                if np.random.rand() < config['tasa_mutacion']:
                    neurona.bias += np.random.uniform(-config['magnitud_mutacion'], config['magnitud_mutacion'])
                # Mutate weights
                # Check if the neuron has weights (i.e., if it has inputs other than itself)
                if hasattr(neurona, 'pesos'):
                    for i in range(len(neurona.pesos)):
                        if np.random.rand() < config['tasa_mutacion']:
                            neurona.pesos[i] += np.random.uniform(-config['magnitud_mutacion'], config['magnitud_mutacion'])

        # Optional: print progress periodically
        if (gen + 1) % 100 == 0 or gen == 1: # Print progress starting from gen 1 and every 100
            print(f"Generation {gen+1}: Current loss: {current_loss:.4f}, Best loss: {best_loss:.4f}")


    print(f"Training finished. Best loss: {best_loss:.4f}")
    return best_red_state # Return the saved best state


def predecir_con_red(entrada_neuronas, ocultas_neuronas, salida_neurona, punto):
    if not isinstance(punto, np.ndarray):
        punto = np.array(punto)

    # Ensure the input point has the correct dimension for the input neurons
    if punto.shape[0] != len(entrada_neuronas):
         raise ValueError(f"Input point dimension mismatch. Expected {len(entrada_neuronas)}, got {punto.shape[0]}")


    for i, val in enumerate(punto):
        entrada_neuronas[i].salida = val
    for h in ocultas_neuronas:
        h.activar()
    salida_neurona.activar()
    return salida_neurona.salida


def vizualizar_frontera(entrada_red, ocultas_red, salida_red, X, y):
    print("\nVisualizando frontera de decisión...")

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    h = 0.02

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # Predict probabilities or output values
    # Ensure the prediction function can handle a batch or iterate over points
    # The current predecir_con_red handles one point at a time, so iterate
    Z_raw = np.array([predecir_con_red(entrada_red, ocultas_red, salida_red, p) for p in tqdm(grid_points, desc="Predicting grid points")])

    # Use the output directly for coloring (if it's a probability/value)
    Z_plot = Z_raw.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    # Use contourf to show the output value across the grid
    # Use a colormap that reflects the output value range (e.g., [0, 1] for sigmoid)
    # You might need to adjust vmax/vmin based on the actual output range of your network
    # Assuming sigmoid output [0, 1], levels and cmap are suitable
    contour = plt.contourf(xx, yy, Z_plot, levels=50, cmap=plt.cm.viridis, alpha=0.8) # Use viridis or another suitable cmap
    plt.colorbar(contour, label="Salida de la Neurona de Salida")


    # Use the actual class labels for coloring the original data points
    # cmap=plt.cm.RdBu is suitable for binary classification
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k', marker='o', s=50, label="Datos Originales")

    plt.title("Frontera/Salida de la Red Neuronal Resonante")
    plt.xlabel("Característica 1 (Normalizada)")
    plt.ylabel("Característica 2 (Normalizada)")
    plt.legend()
    plt.show()

# --- Main execution for training and visualization ---

# 1. Create and Normalize Data
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1, random_state=42)
# Ensure normalization is done correctly, standardizing features independently
scaler = StandardScaler()
X = scaler.fit_transform(X)

# --- Model Configuration ---
model_config = {
    'num_neuronas_entrada': 2, # Matches X.shape[1]
    'num_neuronas_ocultas': 7, # Number of hidden neurons
    'fuerza_ocultas': 0.5,
    'amplificacion_ocultas': 1.0,
    'fuerza_salida': 0.7, # Threshold for the output neuron
    'amplificacion_salida': 2.0,
    'n_generaciones': 5000, # Increase generations for better training
    'tasa_mutacion': 0.1, # Probability of mutation
    'magnitud_mutacion': 0.05 # Magnitude of mutation
}


# Train the network
print("Starting training...")
# Ensure all necessary imports and definitions are available before calling entrenar_con_hiperparametros
# This includes NeuronaResonante, crear_red, evaluar_red, copiar_estado_red, sigmoide, numpy, math
best_network = entrenar_con_hiperparametros(X, y, model_config)

# Visualize the decision boundary/output
if best_network:
    entrada_entrenada, ocultas_entrenadas, salida_entrenada = best_network
    vizualizar_frontera(entrada_entrenada, ocultas_entrenadas, salida_entrenada, X, y)
else:
    print("Training failed. No best network found.")

In [ ]:
import matplotlib.pyplot as plt
import random

# 1. Definir Neurona (posición)
class Neurona:
    def __init__(self, id, x, y):
        self.id = id
        self.x = x
        self.y = y
        self.conexiones = [] # Lista de IDs de neuronas conectadas

# 2. Crear Neuronas
num_neuronas = 20
neuronas = []
for i in range(num_neuronas):
    x = random.uniform(0, 10) # Posición aleatoria en el rango [0, 10]
    y = random.uniform(0, 10)
    neuronas.append(Neurona(i, x, y))

# 3. Definir Conexiones (ejemplo: conexiones aleatorias simples)
for i in range(num_neuronas):
    for j in range(i + 1, num_neuronas): # Evitar duplicados y auto-conexiones
        if random.random() < 0.2: # 20% de probabilidad de conexión
            neuronas[i].conexiones.append(neuronas[j].id)
            neuronas[j].conexiones.append(neuronas[i].id) # Conexión bidireccional

# 4. Visualizar
fig, ax = plt.subplots()

# Dibujar neuronas
for n in neuronas:
    ax.plot(n.x, n.y, 'o', color='skyblue', markersize=10) # Círculos para neuronas

# Dibujar conexiones
for n1 in neuronas:
    for conectado_id in n1.conexiones:
        n2 = next((n for n in neuronas if n.id == conectado_id), None)
        if n2:
            ax.plot([n1.x, n2.x], [n1.y, n2.y], '-', color='blue', linewidth=0.5) # Líneas para conexiones
            # Opcional: añadir puntos luminosos en las conexiones
            # ax.plot((n1.x + n2.x) / 2, (n1.y + n2.y) / 2, '*', color='yellow', markersize=5)


ax.set_title("Simulación de Conexiones Neuronales Espaciales")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal', adjustable='box')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# --- Paso 1: Simular 100 neuronas (cada una con 10 pesos y 1 bias)
n_neuronas = 100
n_entradas = 10

# Pesos y bias aleatorios
pesos = np.random.randn(n_neuronas, n_entradas)
bias = np.random.randn(n_neuronas, 1)

# --- Paso 2: Entrada común para todas las neuronas
entrada = np.random.randn(n_entradas, 1)

# --- Paso 3: Calcular salidas (suma ponderada + ReLU)
activaciones = np.maximum(0, pesos @ entrada + bias)  # ReLU

# --- Paso 4: Usar los pesos+bias como representación de la neurona
vectores_neuronas = np.hstack([pesos, bias])  # cada fila representa una neurona

# --- Paso 5: Reducción a 3D con PCA
pca = PCA(n_components=3)
espacio_3d = pca.fit_transform(vectores_neuronas)

# --- Paso 6: Visualización
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(espacio_3d[:,0], espacio_3d[:,1], espacio_3d[:,2], c='blue')

ax.set_title("Neuronas en espacio reducido 3D (PCA)")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.set_zlabel("Componente 3")
plt.show()

In [ ]:
import numpy as np

# Definición de la clase Neurona
class Neurona:
    def __init__(self, pesos, umbral):
        self.pesos = np.array(pesos)
        self.umbral = umbral
        self.salida = 0

    def activar(self, entradas):
        # Suma ponderada de las entradas
        suma_ponderada = np.dot(entradas, self.pesos)
        # Función de activación (escalón)
        if suma_ponderada >= self.umbral:
            self.salida = 1
        else:
            self.salida = 0
        return self.salida

# Crear dos neuronas
# Neurona 1: Toma dos entradas
neurona1 = Neurona(pesos=[0.5, 0.5], umbral=0.7)

# Neurona 2: Toma la salida de la neurona 1 como una de sus entradas
# y otra entrada externa
neurona2 = Neurona(pesos=[0.6, 0.4], umbral=0.8)

# Entradas para la simulación
entrada_externa_n1_a = 0.8
entrada_externa_n1_b = 0.3
entrada_externa_n2 = 0.9

# Simulación de la conexión
print(f"Entradas para Neurona 1: [{entrada_externa_n1_a}, {entrada_externa_n1_b}]")
salida_neurona1 = neurona1.activar([entrada_externa_n1_a, entrada_externa_n1_b])
print(f"Salida de Neurona 1: {salida_neurona1}")

# La salida de la Neurona 1 es una entrada para la Neurona 2
print(f"Entradas para Neurona 2: [Salida_Neurona1={salida_neurona1}, Entrada_Externa_N2={entrada_externa_n2}]")
salida_neurona2 = neurona2.activar([salida_neurona1, entrada_externa_n2])
print(f"Salida de Neurona 2: {salida_neurona2}")

In [ ]:
# prompt: Amarillo rojo azul, a cada una según su característica o tipo se le da un color único vizualizar 2D import numpy as np
# # Sigmoid function and its derivative
# def sigmoid(x):
#     return 1 / (1 + np.exp(-x))
# def sigmoid_derivative(x):
#     return x * (1 - x)
# # --- Configuración de la Red Neuronal ---
# # Pesos y sesgos (inicializados aleatoriamente o predefinidos)
# # Pesos de la capa de entrada a la capa oculta
# weights_input_hidden = np.array([[0.5]]) # Una entrada, una neurona oculta
# bias_hidden = np.array([0.1])
# # Pesos de la capa oculta a la capa de salida
# weights_hidden_output = np.array([[0.6]]) # Una neurona oculta, una salida
# bias_output = np.array([0.2])
# # Tasa de aprendizaje
# learning_rate = 0.1
# # Número de épocas para el entrenamiento (solo un ejemplo, no se entrena aquí)
# epochs = 1000
# # --- Propagación hacia adelante (Forward Propagation) ---
# def forward_propagation(input_data):
#     # Capa oculta
#     # weighted_sum_hidden = np.dot(input_data, weights_input_hidden) + bias_hidden
#     # Aquí es más simple ya que es 1x1 la matriz de pesos
#     weighted_sum_hidden = input_data[0] * weights_input_hidden[0][0] + bias_hidden[0]
#     output_hidden = sigmoid(weighted_sum_hidden)
#     # Capa de salida
#     # weighted_sum_output = np.dot(output_hidden, weights_hidden_output) + bias_output
#     # Aquí es más simple ya que es 1x1 la matriz de pesos
#     weighted_sum_output = output_hidden * weights_hidden_output[0][0] + bias_output[0]
#     output_final = sigmoid(weighted_sum_output)
#     return output_final, output_hidden, weighted_sum_hidden, weighted_sum_output
# # --- Ejemplo de uso ---
# # Datos de entrada de ejemplo
# input_data = np.array([0.7]) # Una única entrada
# print(f"Entrada: {input_data}")
# # Realizar la propagación hacia adelante
# final_output, hidden_output, _, _ = forward_propagation(input_data)
# print(f"Salida de la capa oculta: {hidden_output}")
# print(f"Salida final de la red: {final_output}")
# # --- Nota: Entrenamiento (Backpropagation) ---
# # Este código solo mu

# Imports for visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np

# --- Data for Visualization ---
# Data from your provided code snippet for visualization
# Using the node_values dictionary from your networkx example
# Mapping these values to specific colors (Amarillo, Rojo, Azul)

# Example data structure: node_id -> value
# You have node_values from a previous cell, assuming it's available
# node_values = {0: 0.7, 1: 0.6, 2: 0.8, 3: 0.68, ...} # Example if not defined

# Let's create a sample node_values dictionary if it's not guaranteed from previous run
if 'node_values' not in locals() or not isinstance(node_values, dict) or len(node_values) == 0:
    print("Creating sample node_values for visualization.")
    num_sample_nodes = 50 # Use a small number for this example
    sample_node_ids = list(range(num_sample_nodes))
    sample_pos = {i: (np.random.rand(), np.random.rand()) for i in sample_node_ids}
    # Assign values in a range that can be mapped to colors
    sample_node_values = {i: 0.65 + (sample_pos[i][1] * (0.80 - 0.65)) for i in sample_node_ids}
    node_values = sample_node_values # Use this sample data

# Mapping values to colors:
# Amarillo: maybe values > 0.75
# Rojo: maybe values between 0.7 and 0.75
# Azul: maybe values <= 0.7

# Define the thresholds for color mapping
threshold_rojo_azul = 0.70
threshold_amarillo_rojo = 0.75

# Assign colors based on the value
color_map = []
for node_id, value in node_values.items():
    if value > threshold_amarillo_rojo:
        color_map.append('yellow')
    elif value > threshold_rojo_azul:
        color_map.append('red')
    else:
        color_map.append('blue')

# You need a NetworkX graph object (G) with nodes corresponding to node_values keys
# If G is not defined from previous cells, create a dummy one for visualization
if 'G' not in locals() or not isinstance(G, nx.Graph) or len(G.nodes) == 0:
    print("Creating a sample NetworkX graph for visualization.")
    G = nx.Graph()
    G.add_nodes_from(node_values.keys())
    # Add some random edges just for visualization structure
    sample_nodes = list(G.nodes())
    num_edges = int(len(sample_nodes) * 1.5) # Add 1.5 times the number of nodes as edges
    for _ in range(num_edges):
        u, v = np.random.choice(sample_nodes, 2, replace=False)
        G.add_edge(u, v)

# You also need node positions (pos) for drawing
# If pos is not defined, create random positions
if 'pos' not in locals() or not isinstance(pos, dict) or len(pos) != len(G.nodes):
     print("Creating sample node positions for visualization.")
     # Ensure pos keys match G.nodes()
     pos = {node: (np.random.rand(), np.random.rand()) for node in G.nodes()}


# Draw the graph
plt.figure(figsize=(10, 10))

# Use the custom color map
nx.draw(G, pos, node_color=color_map, with_labels=False, node_size=50, edge_color='gray', alpha=0.8)

# Add a title
plt.title("Visualización 2D de Nodos con Colores: Azul, Rojo, Amarillo")

# Create a custom legend or color bar to explain the colors
# This requires creating dummy artists or a manual color bar
# For a simple visualization, a manual legend is easier
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label='Azul (<= {:.2f})'.format(threshold_rojo_azul),
               markerfacecolor='blue', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Rojo ({:.2f} - {:.2f})'.format(threshold_rojo_azul, threshold_amarillo_rojo),
               markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Amarillo (> {:.2f})'.format(threshold_amarillo_rojo),
               markerfacecolor='yellow', markersize=10)
]

# Position the legend outside the plot if needed
plt.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1))

plt.axis('off') # Hide axes
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# --- Configuración de las neuronas (ejemplo) ---
# Puedes ajustar estos números según tu red real
num_input_neurons = 10    # Cantidad de neuronas de entrada
num_hidden_neurons = 5    # Cantidad de neuronas en la capa oculta
num_output_neurons = 1    # Cantidad de neuronas de salida
embedding_dim = 768       # Dimensión del embedding de cada "palabra" (neurona de entrada)

# --- Generación de embeddings de ejemplo ---
# En un caso real, estas serían las incrustaciones de palabras reales
# para las neuronas de entrada y las características extraídas para las ocultas/salida.
# Aquí se generan aleatoriamente para demostrar la visualización.
input_embeddings = np.random.rand(num_input_neurons, embedding_dim)
hidden_embeddings = np.random.rand(num_hidden_neurons, embedding_dim)
output_embeddings = np.random.rand(num_output_neurons, embedding_dim)

# Concatenar todos los embeddings para procesarlos juntos
all_embeddings = np.concatenate([input_embeddings, hidden_embeddings, output_embeddings])

# --- Asignación de tipos de neuronas y colores ---
neuron_types = ['input'] * num_input_neurons + \
               ['hidden'] * num_hidden_neurons + \
               ['output'] * num_output_neurons

# Diccionario para mapear tipos a colores
color_map = {
    'input': 'green',    # Neuronas de entrada
    'hidden': 'orange',  # Neuronas ocultas
    'output': 'red'      # Neuronas de salida
}
colors = [color_map[neuron_type] for neuron_type in neuron_types]

# --- Reducción de Dimensionalidad con t-SNE ---
# t-SNE es una técnica para reducir datos de alta dimensión a 2D o 3D
# manteniendo la similitud entre los puntos.
# `perplexity` debe ser menor que el número total de muestras.
tsne = TSNE(n_components=2, random_state=0, perplexity=min(5, len(all_embeddings) - 1))
reduced_embeddings = tsne.fit_transform(all_embeddings)

# --- Visualización 2D de las neuronas ---
plt.figure(figsize=(12, 10)) # Tamaño del gráfico

# Dibuja los puntos (neuronas)
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=colors, s=150, alpha=0.8)

# Añade etiquetas para el tipo de neurona
for i, neuron_type in enumerate(neuron_types):
    plt.annotate(neuron_type,
                 xy=(reduced_embeddings[i, 0], reduced_embeddings[i, 1]),
                 xytext=(8, 4), # Desplazamiento del texto
                 textcoords='offset points',
                 ha='left', va='bottom',
                 fontsize=9, color='dimgray')

# Título y etiquetas de los ejes
plt.title('Visualización 2D Espacial de Neuronas (Reducción con t-SNE)', fontsize=16)
plt.xlabel('Dimensión 1 (t-SNE)', fontsize=12)
plt.ylabel('Dimensión 2 (t-SNE)', fontsize=12)

# Crear una leyenda manual
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label='Entrada (Verde)',
                          markerfacecolor='green', markersize=10),
                   Line2D([0], [0], marker='o', color='w', label='Oculta (Naranja)',
                          markerfacecolor='orange', markersize=10),
                   Line2D([0], [0], marker='o', color='w', label='Salida (Roja)',
                          markerfacecolor='red', markersize=10)]
plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")

plt.grid(True, linestyle='--', alpha=0.6) # Añadir una cuadrícula sutil
plt.show()

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np # Import numpy

# --- Paso 1: Cargar modelo y tokenizer ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()  # No queremos entrenamiento

# --- Paso 2: Elegir palabras semánticamente cercanas ---
# Aumentamos la lista de palabras para tener más de 2 muestras
palabras = ["dog", "puppy", "cat", "kitten", "man", "woman", "king", "queen"]

# --- Paso 3: Obtener embeddings ---
embeddings = []

for palabra in palabras:
    inputs = tokenizer(palabra, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    # Tomamos el embedding del primer token real (ignoramos [CLS], usamos primera subpalabra)
    # Aseguramos que el tensor está en CPU antes de convertir a numpy
    token_id = 1
    # Verificamos que token_id es válido para la secuencia tokenizada
    if token_id < inputs["input_ids"].shape[1]:
        embedding = outputs.last_hidden_state[0, token_id, :].cpu().numpy()  # shape: (768,)
        embeddings.append(embedding)
    else:
        print(f"Warning: token_id {token_id} out of range for word '{palabra}'. Skipping.")


# --- Paso 4: Reducir a 2D con t-SNE ---
# Convertir la lista de embeddings a un array de NumPy
embeddings_np = np.array(embeddings)

# `perplexity` debe ser menor que el número de muestras.
# Ahora tenemos más muestras, podemos usar una perplexity típica (e.g., 5)
# Asegurarse de que n_samples > perplexity
if embeddings_np.shape[0] < 5:
    # Si por alguna razón se cargaron menos de 5 embeddings, ajustamos perplexity
    perplexity_value = max(2, embeddings_np.shape[0] - 1)
    print(f"Not enough samples ({embeddings_np.shape[0]}) for perplexity 5. Using perplexity={perplexity_value}")
else:
    perplexity_value = 5

tsne = TSNE(n_components=2, perplexity=perplexity_value, random_state=0)
# Usar el array de NumPy para fit_transform
embeddings_2d = tsne.fit_transform(embeddings_np)


# --- Paso 5: Visualización ---
plt.figure(figsize=(10, 8)) # Aumentar tamaño para más puntos
for i, palabra in enumerate(palabras):
    # Asegurarse de que el índice i es válido para embeddings_2d
    if i < embeddings_2d.shape[0]:
        x, y = embeddings_2d[i]
        plt.scatter(x, y, s=150, label=palabra)
        plt.text(x + 0.1, y + 0.1, palabra, fontsize=12)
    else:
        print(f"Warning: Index {i} out of bounds for embeddings_2d. Skipping annotation for '{palabra}'.")


plt.title("Visualización 2D de embeddings BERT", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# 1. Cargar el modelo y tokenizer de BERT
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

# 2. Tu texto de ejemplo
texto = "La casa es grande y tiene un jardín hermoso con flores."
tokens = tokenizer.tokenize(texto)
# Añadir los tokens especiales [CLS] y [SEP] que BERT espera
tokens_bert = ['[CLS]'] + tokens + ['[SEP]']
# Mapear tokens a IDs de entrada
input_ids = tokenizer.convert_tokens_to_ids(tokens_bert)
input_ids = torch.tensor([input_ids]) # Convertir a tensor de PyTorch

# 3. Obtener los embeddings de BERT
with torch.no_grad(): # No necesitamos calcular gradientes para esto
    outputs = model(input_ids)
    # embeddings[0] contiene los hidden states para cada token en la última capa.
    # Cada token tiene un embedding de 768 dimensiones.
    word_embeddings = outputs.last_hidden_state[0] # Eliminar el batch dimension

# 4. Preparar los datos para t-SNE (excluir [CLS] y [SEP] para claridad en la visualización de palabras)
# Nota: Puedes incluir [CLS] y [SEP] si quieres ver cómo se agrupan.
embeddings_a_visualizar = word_embeddings[1:-1].numpy() # Convertir a NumPy array
palabras_a_visualizar = tokens # Las palabras originales sin [CLS]/[SEP]

# 5. Aplicar t-SNE para reducir a 2 dimensiones (puedes cambiar n_components a 3 para 3D)
tsne = TSNE(n_components=2, random_state=42, perplexity=len(palabras_a_visualizar)-1 if len(palabras_a_visualizar) > 1 else 1)
embeddings_2d = tsne.fit_transform(embeddings_a_visualizar)

# 6. Imprimir las "coordenadas" y visualizar
print("Coordenadas 2D (x, y) de las palabras:")
for i, (x, y) in enumerate(embeddings_2d):
    print(f"Palabra: '{palabras_a_visualizar[i]}', X: {x:.4f}, Y: {y:.4f}")

# 7. Visualización (opcional, pero útil)
plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1])
for i, txt in enumerate(palabras_a_visualizar):
    plt.annotate(txt, (embeddings_2d[i, 0], embeddings_2d[i, 1]), textcoords="offset points", xytext=(5,5), ha='center')
plt.title("Visualización 2D de Embeddings de BERT (t-SNE)")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.grid(True)
plt.show()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Cargar un modelo BERT (ej. 'bert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased") # Un modelo español
model = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

text = "La casa es grande y bonita."
tokens = tokenizer.tokenize(text)
# Añadir tokens especiales CLS y SEP para BERT
indexed_tokens = tokenizer.convert_tokens_to_ids(["[CLS]"] + tokens + ["[SEP]"])
segments_ids = [1] * len(indexed_tokens) # Segmentos (para una sola oración)

# Convertir a tensores de PyTorch
tokens_tensor = torch.tensor([indexed_tokens])
segments_tensors = torch.tensor([segments_ids])

# Obtener los embeddings
with torch.no_grad(): # Desactivar el cálculo de gradientes para inferencia
    outputs = model(tokens_tensor, segments_tensors)
    # Los embeddings están en outputs[0], para cada token
    # Tomamos el embedding de la última capa
    last_hidden_states = outputs.last_hidden_state

# last_hidden_states[0] contiene los embeddings de 768 dimensiones para cada token en la oración
# [CLS], La, casa, es, grande, y, bonita, ., [SEP]
embeddings_de_palabras = last_hidden_states[0].numpy()
palabras_originales = tokenizer.convert_ids_to_tokens(indexed_tokens)

print(f"Número de tokens: {len(palabras_originales)}")
print(f"Dimensiones de cada embedding: {embeddings_de_palabras.shape[1]}")

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Ejemplo para reducir a 2 dimensiones (X, Y)
# Puedes cambiar n_components a 3 para obtener (X, Y, Z)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(embeddings_de_palabras) - 1))
embeddings_2d = tsne.fit_transform(embeddings_de_palabras)

# Crear un DataFrame para facilitar la visualización
df_embeddings = pd.DataFrame(embeddings_2d, columns=['X', 'Y'])
df_embeddings['Palabra'] = palabras_originales
df_embeddings['Posición'] = range(len(palabras_originales)) # Añadir la posición explícita

# Visualizar
plt.figure(figsize=(10, 8))
plt.scatter(df_embeddings['X'], df_embeddings['Y'])
for i, row in df_embeddings.iterrows():
    plt.annotate(f"{row['Palabra']} ({row['Posición']})", (row['X'], row['Y']), textcoords="offset points", xytext=(5,5), ha='center')

plt.title('Visualización 2D de Embeddings de BERT (t-SNE)')
plt.xlabel('Dimensión X')
plt.ylabel('Dimensión Y')
plt.grid(True)
plt.show()

# Si quieres un resultado directo de X, Y, Z (o X, Y en este caso)
# df_embeddings[['X', 'Y']] te daría esas coordenadas.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt # Importar matplotlib para visualización

#**1. Obtener Embeddings de BERT y Coordenadas Reducidas:**

def obtener_embeddings_y_coordenadas(texto, modelo_nombre="dccuchile/bert-base-spanish-wwm-cased", n_componentes_tsne=2, random_state=42):
    tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
    model = AutoModel.from_pretrained(modelo_nombre)

    tokens = tokenizer.tokenize(texto)
    # Asegúrate de que los tokens especiales están escritos correctamente, sin barras invertidas
    tokens_bert = ['[CLS]'] + tokens + ['[SEP]']
    input_ids = tokenizer.convert_tokens_to_ids(tokens_bert)
    input_ids = torch.tensor([input_ids])

    with torch.no_grad():
        outputs = model(input_ids)
        word_embeddings = outputs.last_hidden_state[0]

    # Excluir [CLS] y [SEP]. Asegúrate de que el slicing es correcto.
    # `embeddings_a_visualizar = word_embeddings\[1:-1].numpy()` tenía una barra invertida extra.
    embeddings_a_visualizar = word_embeddings[1:-1].numpy()
    palabras = tokens

    # Ajustar perplexity si el número de muestras es pequeño
    if embeddings_a_visualizar.shape[0] > n_componentes_tsne:
        perplexity_value = min(30, embeddings_a_visualizar.shape[0] - 1)
        # Ensure perplexity is at least 1 if there's at least one sample
        perplexity_value = max(1, perplexity_value)
        tsne = TSNE(n_components=n_componentes_tsne, random_state=random_state, perplexity=perplexity_value)
        coordenadas_reducidas = tsne.fit_transform(embeddings_a_visualizar)
    else:
        # Manejar casos con pocas palabras: devuelve ceros o un array vacío dependiendo de la necesidad.
        # Crear un array de ceros con la forma esperada
        coordenadas_reducidas = np.zeros((embeddings_a_visualizar.shape[0], n_componentes_tsne))
        if embeddings_a_visualizar.shape[0] == 0:
             print("Advertencia: No se extrajeron embeddings de palabras (después de excluir [CLS]/[SEP]). No se puede aplicar t-SNE.")


    return palabras, embeddings_a_visualizar, coordenadas_reducidas

#**2. Definir una Red Neuronal Simple:**

class MiRed(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MiRed, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

#**3. Preparar los Datos y Entrenar (Ejemplo Conceptual):**

# Texto de ejemplo
texto_ejemplo = "El perro ladra fuerte porque ve un gato negro."
palabras, embeddings_bert, coordenadas_tsne = obtener_embeddings_y_coordenadas(texto_ejemplo)

# --- Decidir cómo usar la información para la red ---
combinar_embedding_con_coordenadas = True # Cambiar a False si solo quieres usar el embedding

datos_de_entrada = []
# Asegurarse de iterar sobre el número de embeddings/palabras disponibles
if len(palabras) > 0:
    for i in range(len(palabras)):
        embedding = embeddings_bert[i]
        if combinar_embedding_con_coordenadas:
            # Asegurarse de que tenemos coordenadas para este índice
            if i < coordenadas_tsne.shape[0]:
                coordenada = coordenadas_tsne[i]
                entrada = np.concatenate((embedding, coordenada))
                datos_de_entrada.append(entrada)
            else:
                 print(f"Advertencia: No hay coordenadas t-SNE disponibles para el índice {i}. Saltando.")
        else:
            datos_de_entrada.append(embedding)

# Solo convertir a tensor si hay datos de entrada
if datos_de_entrada:
    datos_de_entrada = torch.tensor(np.array(datos_de_entrada), dtype=torch.float32)

    # --- Crear etiquetas de ejemplo (necesario para el entrenamiento supervisado) ---
    # Esto dependerá de tu tarea. Aquí solo se crean etiquetas aleatorias.
    # Asegurarse de que el número de etiquetas coincide con el número de muestras de entrada.
    etiquetas_ejemplo = torch.randint(0, 2, (datos_de_entrada.shape[0],)).float().unsqueeze(1) # Ejemplo binario

    # --- Definir la red ---
    input_dim_red = datos_de_entrada.shape[1] # Obtener la dimensión de entrada directamente de los datos
    hidden_dim_red = 128
    output_dim_red = 1 # Para clasificación binaria

    modelo = MiRed(input_dim_red, hidden_dim_red, output_dim_red)
    criterio = nn.BCEWithLogitsLoss() # Ejemplo para clasificación binaria
    optimizador = optim.Adam(modelo.parameters(), lr=0.01)

    # --- Bucle de entrenamiento (muy simplificado) ---
    epocas = 10
    for epoca in range(epocas):
        optimizador.zero_grad()
        salida = modelo(datos_de_entrada)
        perdida = criterio(salida, etiquetas_ejemplo)
        perdida.backward()
        optimizador.step()
        print(f'Época {epoca+1}, Pérdida: {perdida.item():.4f}')

    # --- (Opcional) Visualizar los embeddings reducidos ---
    # Solo visualizar si hay coordenadas t-SNE disponibles
    if coordenadas_tsne.shape[0] > 0:
        plt.figure(figsize=(10, 6))
        plt.scatter(coordenadas_tsne[:, 0], coordenadas_tsne[:, 1])
        for i, palabra in enumerate(palabras):
             # Asegurarse de que el índice es válido para las coordenadas
            if i < coordenadas_tsne.shape[0]:
                plt.annotate(palabra, xy=coordenadas_tsne[i], xytext=(5, 2),
                             textcoords='offset points', ha='right', va='bottom')
        plt.title('Visualización t-SNE de Embeddings')
        plt.xlabel('Dimensión 1')
        plt.ylabel('Dimensión 2')
        plt.show()
    else:
        print("No se pudieron visualizar los embeddings t-SNE ya que no se generaron coordenadas.")

else:
    print("No se pudieron preparar datos de entrada para la red neuronal.")

#nx.draw_networkx_nodes

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors # Necesario para mcolors.Normalize

# Datos de ejemplo: Nodos con posiciones y un valor asociado para el color
num_nodes = 2000 # Número de nodos (ajustar para densidad)
radius = 0.12 # Radio para conexiones (ajustar para conectividad) - AUMENTADO
np.random.seed(42) # para reproducibilidad

pos = {}
node_values = {}
nodes_list = []

for i in range(num_nodes):
    node_id = i
    nodes_list.append(node_id)
    # Generar posiciones aleatorias dentro del cuadrado [0, 1] x [0, 1]
    pos[node_id] = (np.random.rand(), np.random.rand())
    # Asignar un valor para el color basado en la coordenada Y para un gradiente vertical
    # ¡Importante! Aquí simulamos que los valores de los nodos están en el rango deseado.
    # Si tus valores reales de nodos no están en este rango, necesitarías un mapeo más complejo.
    # Para este ejemplo, ajustamos node_values para que se distribuyan en el rango 0.65-0.80
    node_values[node_id] = 0.65 + (pos[node_id][1] * (0.80 - 0.65)) # Escalar Y para el rango deseado

# Crear el grafo y añadir los nodos con sus posiciones
G = nx.Graph()
G.add_nodes_from(nodes_list)

# Añadir aristas basado en la distancia (solo para nodos cercanos)
for i in range(num_nodes):
    for j in range(i + 1, num_nodes):
        node1 = nodes_list[i]
        node2 = nodes_list[j]
        p1 = np.array(pos[node1])
        p2 = np.array(pos[node2])
        distance = np.linalg.norm(p1 - p2)
        if distance < radius:
            G.add_edge(node1, node2)

# Obtener los valores para colorear los nodos en un array
node_colors_values_array = np.array([node_values[node] for node in G.nodes()])

# Definir el rango deseado para la barra de color
min_val_cbar = 0.65
max_val_cbar = 0.80

# Crear una normalización específica para el rango 0.65 a 0.80
norm = mcolors.Normalize(vmin=min_val_cbar, vmax=max_val_cbar)

# Usar un colormap que se parezca a blue/purple to red/yellow.
cmap = plt.cm.plasma

# Mapear los valores normalizados al colormap
node_colors_mapped = cmap(norm(node_colors_values_array))

# Dibujar el grafo
plt.figure(figsize=(10, 10))
ax = plt.gca() # Obtener los ejes actuales
ax.set_facecolor('white') # Fondo blanco
ax.set_aspect('equal', adjustable='box') # Asegurar que el cuadrado se vea cuadrado

# Dibujar las aristas
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', alpha=0.1)

# Dibujar los nodos
nx.draw_networkx_nodes(G, pos, node_color=node_colors_mapped, node_size=15, ax=ax, linewidths=0.1)

# Crear la barra de color vertical a la derecha
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([]) # Se necesita un array, pero podemos dejarlo vacío si la normalización está bien definida
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.02, pad=0.03)

# Establecer los ticks y etiquetas de la barra de color en el rango deseado
cbar_ticks = np.linspace(min_val_cbar, max_val_cbar, num=6) # Genera 6 ticks uniformes en el rango
cbar.set_ticks(cbar_ticks)
cbar.set_ticklabels([f'{t:.2f}' for t in cbar_ticks])
cbar.set_label('Valor del Nodo', fontsize=12)

plt.axis('off') # Ocultar los ejes
plt.title('Red de Nodos con Barra de Color (0.65 - 0.80)', fontsize=14)
plt.tight_layout()
plt.show()

#GradientReversalFunction

In [ ]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim

# --- Capa Anti-Gradiente (GRL) ---
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_val=1.0):
        super().__init__()
        self.lambda_val = torch.tensor(lambda_val, requires_grad=False)
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)

# --- Modelo Simple con GRL ---
class SimpleAntiGradientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU()
        )
        self.anti_grad_layer = GradientReversalLayer(lambda_val=1.0)
        self.classifier = nn.Sequential(
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature_extractor(x)
        processed = self.anti_grad_layer(features)
        return self.classifier(processed)

# --- Datos XOR ---
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
y = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float32)

# --- Entrenamiento ---
model = SimpleAntiGradientModel()
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

for epoch in range(2000):
    optimizer.zero_grad()
    preds = model(X)
    loss = loss_fn(preds, y)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        print(f"Época {epoch}, Pérdida: {loss.item():.4f}")

# --- Guardar modelo ---
torch.save(model.state_dict(), "/content/antigrad_model.pt")
print("✅ Modelo guardado en /content/antigrad_model.pt")

In [ ]:
# load_model.py
import torch
import torch.nn as nn

# --- Reusar Modelo y GRL ---
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_val=1.0):
        super().__init__()
        self.lambda_val = torch.tensor(lambda_val, requires_grad=False)
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)

class SimpleAntiGradientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU()
        )
        self.anti_grad_layer = GradientReversalLayer(lambda_val=1.0)
        self.classifier = nn.Sequential(
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature_extractor(x)
        processed = self.anti_grad_layer(features)
        return self.classifier(processed)

# --- Cargar modelo ---
model = SimpleAntiGradientModel()
model.load_state_dict(torch.load("/content/antigrad_model.pt"))
model.eval()

# --- Evaluación ---
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
y = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float32)

with torch.no_grad():
    preds = model(X)
    print("✅ Predicciones:")
    print(f"Predicciones:\n{preds.round()}")
    print(f"Etiquetas reales:\n{y}")

#ReversePredictionNet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- Configuración ---
config = {
    "lr": 1e-2,
    "epochs": 10000,
    "model_path": "/content/reverse_causal.pth"
}

# --- Generación de datos causales ---
def generate_data(n):  # Y = 2X + 5 + ruido
    X = torch.randn(n, 1) * 10
    Y = 2 * X + 5 + torch.randn(n, 1) * 2
    return X, Y

X_train, Y_train = generate_data(1000)

# --- Modelo inverso ---
class ReversePredictionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, y): return self.model(y)

model = ReversePredictionNet()
optimizer = optim.Adam(model.parameters(), lr=config["lr"])
loss_fn = nn.MSELoss()

# --- Entrenamiento ---
for epoch in range(config["epochs"]):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(Y_train), X_train)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 500 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

torch.save(model.state_dict(), config["model_path"])
print("✅ Modelo guardado:", config["model_path"])

In [ ]:
import torch

# --- Configuración ---
config = {
    "model_path": "/content/reverse_causal.pth"
}

# --- Cargar modelo ---
model = ReversePredictionNet()
model.load_state_dict(torch.load(config["model_path"]))
model.eval()

# --- Generar datos de prueba ---
def generate_data(n):
    X = torch.randn(n, 1) * 10
    Y = 2 * X + 5 + torch.randn(n, 1) * 2
    return X, Y

X_test, Y_test = generate_data(200)

# --- Evaluación ---
with torch.no_grad():
    preds = model(Y_test)
    loss = torch.nn.functional.mse_loss(preds, X_test)
    print(f"🔍 Pérdida en test: {loss.item():.4f}")
    for i in range(5):
        print(f"Y: {Y_test[i].item():.2f} → X real: {X_test[i].item():.2f}, X predicho: {preds[i].item():.2f}")

#Graphus Nodea Visualis

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def VisualizarGrafoNodos(
    num_nodes: int,
    radius: float,
    min_val_cbar: float,
    max_val_cbar: float,
    cmap_name: str = 'plasma',
    figure_size: tuple = (10, 10),
    title: str = 'Red de Nodos con Barra de Color',
    seed: int = None
):
    """
    Graphus Nodea Visualis: Genera y visualiza un grafo de nodos en 2D,
    coloreados según un valor asociado y con una barra de color personalizada.

    Args:
        num_nodes (int): Número de nodos en el grafo.
        radius (float): Radio máximo para conectar nodos (distancia euclidiana).
        min_val_cbar (float): Valor mínimo para la barra de color y normalización.
        max_val_cbar (float): Valor máximo para la barra de color y normalización.
        cmap_name (str): Nombre del colormap de Matplotlib a usar (ej. 'plasma', 'viridis', 'cividis').
        figure_size (tuple): Tupla (ancho, alto) para el tamaño de la figura.
        title (str): Título del gráfico.
        seed (int, optional): Semilla para la reproducibilidad de la generación de nodos.
                              Si es None, los nodos se generarán aleatoriamente cada vez.
    """
    if seed is not None:
        np.random.seed(seed)

    pos = {}
    node_values = {}
    nodes_list = []

    for i in range(num_nodes):
        node_id = i
        nodes_list.append(node_id)
        pos[node_id] = (np.random.rand(), np.random.rand())
        # Escalar los valores de los nodos para que se ajusten al rango deseado
        # Aquí se asume que los valores se derivan de la coordenada Y,
        # pero en una aplicación real, 'node_values' podría ser una entrada de la función.
        node_values[node_id] = min_val_cbar + (pos[node_id][1] * (max_val_cbar - min_val_cbar))

    G = nx.Graph()
    G.add_nodes_from(nodes_list)

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            node1 = nodes_list[i]
            node2 = nodes_list[j]
            p1 = np.array(pos[node1])
            p2 = np.array(pos[node2])
            distance = np.linalg.norm(p1 - p2)
            if distance < radius:
                G.add_edge(node1, node2)

    node_colors_values_array = np.array([node_values[node] for node in G.nodes()])

    norm = mcolors.Normalize(vmin=min_val_cbar, vmax=max_val_cbar)
    cmap = plt.cm.get_cmap(cmap_name) # Obtener el colormap por nombre

    node_colors_mapped = cmap(norm(node_colors_values_array))

    plt.figure(figsize=figure_size)
    ax = plt.gca()
    ax.set_facecolor('white')
    ax.set_aspect('equal', adjustable='box')

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', alpha=0.1)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors_mapped, node_size=15, ax=ax, linewidths=0.1)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.02, pad=0.03)

    cbar_ticks = np.linspace(min_val_cbar, max_val_cbar, num=6)
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels([f'{t:.2f}' for t in cbar_ticks])
    cbar.set_label('Valor del Nodo', fontsize=12)

    plt.axis('off')
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

##vector de condición

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NormalizatioAdaptiva(nn.Module):
    """
    Normalización Adaptativa basada en un vector de condición.
    Ajusta dinámicamente los parámetros de escala (gamma) y bias (beta)
    en función de una entrada condicional, similar a SPADE o AdaIN.
    """
    def __init__(self, num_features: int, cond_dim: int):
        """
        Inicializa el módulo de normalización adaptativa.

        Args:
            num_features (int): Número de canales de las características de entrada.
            cond_dim (int): Dimensión del vector de condición.
        """
        super().__init__()
        # InstanceNorm2d para normalizar las características de forma independiente para cada instancia
        self.norm = nn.InstanceNorm2d(num_features, affine=False)
        # Capas lineales para proyectar el vector de condición a gamma y beta
        self.gamma_proj = nn.Linear(cond_dim, num_features)
        self.beta_proj = nn.Linear(cond_dim, num_features)

    def forward(self, x: torch.Tensor, cond_vector: torch.Tensor) -> torch.Tensor:
        """
        Aplica la normalización adaptativa a las activaciones.

        Args:
            x (torch.Tensor): Activaciones de entrada [batch, channels, H, W].
            cond_vector (torch.Tensor): Vector de condición [batch, cond_dim].

        Returns:
            torch.Tensor: Activaciones normalizadas y adaptadas.
        """
        # Normalizar las características de entrada
        out = self.norm(x)

        # Predecir gamma y beta a partir del vector de condición
        # Se añaden dimensiones extra para permitir el broadcasting en las operaciones de multiplicación/suma
        gamma = self.gamma_proj(cond_vector).unsqueeze(-1).unsqueeze(-1)  # [batch, channels, 1, 1]
        beta = self.beta_proj(cond_vector).unsqueeze(-1).unsqueeze(-1)    # [batch, channels, 1, 1]

        # Aplicar escala (1 + gamma) y bias (beta) a las activaciones normalizadas
        out = out * (1 + gamma) + beta
        return out

##ConditionalFusionBlock (Nexus Condicionalis)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# La clase NormalizatioAdaptiva debe estar definida o importada previamente para que NexusCondicionalis funcione.
# Asumimos que 'NormalizatioAdaptiva' está disponible en el mismo ámbito o ha sido importada.

class NexusCondicionalis(nn.Module):
    """
    Bloque de Fusión Condicional que combina características de la red
    con múltiples vectores de condición utilizando Normalización Adaptativa.
    """
    def __init__(self, in_channels: int, out_channels: int, img_id_cond_dim: int, text_cond_dim: int):
        """
        Inicializa el Bloque de Fusión Condicional.

        Args:
            in_channels (int): Número de canales de entrada.
            out_channels (int): Número de canales de salida.
            img_id_cond_dim (int): Dimensión del vector de condición de imagen/ID.
            text_cond_dim (int): Dimensión del vector de condición de texto.
        """
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # Normalización adaptativa para la condición de imagen/ID
        self.norm_img_id = NormalizatioAdaptiva(out_channels, img_id_cond_dim)
        # Normalización adaptativa para la condición de texto
        self.norm_text = NormalizatioAdaptiva(out_channels, text_cond_dim)

    def forward(self, x: torch.Tensor, img_id_vector: torch.Tensor, text_cond_vector: torch.Tensor) -> torch.Tensor:
        """
        Procesa las características de entrada con las condiciones.

        Args:
            x (torch.Tensor): Mapa de características de la capa anterior [batch, in_channels, H, W].
            img_id_vector (torch.Tensor): Vector de características de imagen + ID [batch, img_id_cond_dim].
            text_cond_vector (torch.Tensor): Vector de características de texto [batch, text_cond_dim].

        Returns:
            torch.Tensor: Mapa de características fusionado condicionalmente.
        """
        out = self.conv(x)

        # Aplicar normalización adaptativa para cada tipo de condición.
        # El orden de aplicación puede ser importante.
        out = self.norm_img_id(out, img_id_vector)
        out = self.norm_text(out, text_cond_vector)

        out = self.relu(out)
        return out

##HCMG (Generatio Multimodalis Hierarchica)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Las clases NormalizatioAdaptiva y NexusCondicionalis deben estar definidas o importadas.
# Asumimos que están disponibles en el mismo ámbito o han sido importadas.

class GeneratioMultimodalisHierarchica(nn.Module):
    """
    Generador Condicionado Multimodal y Jerárquico.
    Un modelo de red neuronal convolucional capaz de generar imágenes
    condicionado por múltiples tipos de entrada (píxeles, ID de imagen, texto).
    """
    def __init__(self, input_image_channels: int, output_image_channels: int,
                 img_id_cond_dim: int, text_cond_dim: int, base_channels: int = 64):
        """
        Inicializa el Generador Condicionado Multimodal y Jerárquico.

        Args:
            input_image_channels (int): Número de canales de la imagen de entrada (ej. 3 para RGB).
            output_image_channels (int): Número de canales de la imagen de salida (ej. 3 para RGB).
            img_id_cond_dim (int): Dimensión del vector de condición de imagen/ID.
            text_cond_dim (int): Dimensión del vector de condición de texto.
            base_channels (int): Número de canales base para las capas internas del generador.
        """
        super().__init__()

        self.initial_conv = nn.Conv2d(input_image_channels, base_channels, kernel_size=3, padding=1)

        # Bloques de Fusión Condicional (encoder)
        self.bfc1 = NexusCondicionalis(base_channels, base_channels * 2, img_id_cond_dim, text_cond_dim)
        self.downsample1 = nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1)

        self.bfc2 = NexusCondicionalis(base_channels * 2, base_channels * 4, img_id_cond_dim, text_cond_dim)
        self.downsample2 = nn.Conv2d(base_channels * 4, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bfc3 = NexusCondicionalis(base_channels * 4, base_channels * 8, img_id_cond_dim, text_cond_dim)

        # Bloques de Fusión Condicional (decoder)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.gen_conv1 = NexusCondicionalis(base_channels * 4, base_channels * 2, img_id_cond_dim, text_cond_dim)

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.gen_conv2 = NexusCondicionalis(base_channels, base_channels, img_id_cond_dim, text_cond_dim)

        self.final_conv = nn.Conv2d(base_channels, output_image_channels, kernel_size=3, padding=1)
        self.tanh = nn.Tanh()

    def forward(self, input_pixels: torch.Tensor, img_id_vector: torch.Tensor, text_cond_vector: torch.Tensor) -> torch.Tensor:
        """
        Define el paso hacia adelante para la generación de la imagen.

        Args:
            input_pixels (torch.Tensor): Imagen de entrada (ej. contorno, imagen borrosa) [batch, channels, H, W].
            img_id_vector (torch.Tensor): Vector de condición para la imagen/ID [batch, img_id_cond_dim].
            text_cond_vector (torch.Tensor): Vector de condición para el texto [batch, text_cond_dim].

        Returns:
            torch.Tensor: La imagen generada [batch, output_image_channels, H, W].
        """
        x = self.initial_conv(input_pixels)

        # Encoder con downsampling
        x = self.bfc1(x, img_id_vector, text_cond_vector)
        x = self.downsample1(x)

        x = self.bfc2(x, img_id_vector, text_cond_vector)
        x = self.downsample2(x)

        x = self.bfc3(x, img_id_vector, text_cond_vector)

        # Decoder con upsampling
        x = self.upsample1(x)
        x = self.gen_conv1(x, img_id_vector, text_cond_vector)

        x = self.upsample2(x)
        x = self.gen_conv2(x, img_id_vector, text_cond_vector)

        # Capa de salida y función de activación (tanh para normalizar la imagen a -1, 1)
        output_image = self.tanh(self.final_conv(x))
        return output_image

##generadores_condicionales.py)

In [ ]:
# En otro archivo Python (ej. main.py)
from generadores_condicionales import NormalizatioAdaptiva, NexusCondicionalis, GeneratioMultimodalisHierarchica

# Luego puedes instanciar y usar tus modelos:
input_channels = 3
output_channels = 3
img_id_cond_dim = 128
text_cond_dim = 768

my_generator = GeneratioMultimodalisHierarchica(input_channels, output_channels, img_id_cond_dim, text_cond_dim)
print(my_generator)

##(ReversioGradientisFunctio)

In [ ]:
import torch

class ReversioGradientisFunctio(torch.autograd.Function):
    """
    ReversioGradientisFunctio: Una función de PyTorch Autograd que
    invierte el signo de los gradientes durante la retropropagación,
    multiplicándolos por un factor lambda.
    """
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float) -> torch.Tensor:
        """
        Paso hacia adelante: pasa la entrada 'x' sin modificar.

        Args:
            x (torch.Tensor): Tensor de entrada.
            lambda_val (float): Factor de escala para la inversión del gradiente.
                                Se guarda para el backward pass.

        Returns:
            torch.Tensor: La entrada 'x' sin cambios.
        """
        ctx.lambda_val = lambda_val # Guarda lambda_val para el backward pass
        return x.view_as(x) # Retorna una vista de x, sin modificar su contenido

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> tuple[torch.Tensor, None]:
        """
        Paso hacia atrás: invierte el signo del gradiente de salida y lo escala.

        Args:
            grad_output (torch.Tensor): El gradiente que fluye hacia atrás desde la capa siguiente.

        Returns:
            tuple[torch.Tensor, None]: El gradiente invertido para 'x' y None para 'lambda_val'
                                       (ya que no calculamos gradientes para este hiperparámetro).
        """
        # Multiplica el gradiente por -lambda_val para invertirlo y escalarlo
        output_grad_x = grad_output * -ctx.lambda_val
        return output_grad_x, None # El segundo None es para el gradiente de lambda_val

##(StratumReversioGradientis)

In [ ]:
import torch
import torch.nn as nn

# La clase ReversioGradientisFunctio debe estar definida o importada previamente.
# Asumimos que está disponible en el mismo ámbito o ha sido importada.

class StratumReversioGradientis(nn.Module):
    """
    StratumReversioGradientis: Una capa de PyTorch que utiliza
    ReversioGradientisFunctio para invertir los gradientes que fluyen a través de ella.
    Comúnmente utilizada en tareas de adaptación de dominio (Domain Adaptation)
    para alinear características o confundir clasificadores de dominio.
    """
    def __init__(self, lambda_val: float = 1.0):
        """
        Inicializa la capa de inversión de gradiente.

        Args:
            lambda_val (float): Factor de escala para la inversión del gradiente.
                                 Debe ser un valor no entrenable.
        """
        super(StratumReversioGradientis, self).__init__()
        # Se guarda lambda_val como un tensor no entrenable.
        self.lambda_val = torch.tensor(lambda_val, requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Paso hacia adelante de la capa: simplemente aplica la función de inversión
        de gradiente a la entrada.

        Args:
            x (torch.Tensor): Tensor de entrada a la capa.

        Returns:
            torch.Tensor: La entrada 'x' sin modificar en el forward pass.
        """
        # Aplica la función estática definida para manejar el comportamiento backward.
        return ReversioGradientisFunctio.apply(x, self.lambda_val)